資料來源:
Invent連結： 
kaggle.com/bhanupratapbiswas/inventory-analysis-case-study
PwC 是 PricewaterhouseCoopers 的縮寫，全球四大會計師事務所（Big Four）之一 。
數據集的業務邏輯設計符合真實審計和財務分析標準，欄位設計反映的是真實企業的採購流程，


數據集結構分析

| 檔案                           | 內容     | 核心欄位                                                                                          |
| ---------------------------- | ------ | --------------------------------------------------------------------------------------------- |
| SalesFINAL12312016.csv       | 銷售交易明細 | inventory_id, store, brand, sales_quantity, sales_dollars, sales_price, sales_date, vendor_no |
| PurchasesFINAL12312016.csv   | 採購入庫明細 | inventory_id, store, brand, purchase_quantity, purchase_price, purchase_date                  |
| BegInvFINAL12312016.csv      | 年初庫存快照 | inventory_id, store, brand, on_hand (期初)                                                      |
| EndInvFINAL12312016.csv      | 年末庫存快照 | inventory_id, store, brand, on_hand (期末)                                                      |
| InvoicePurchases12312016.csv | 採購發票   | vendor_no, invoice_date, quantity, dollars                                                    |
| 2017PurchasePricesDec.csv    | 參考採購價格 | brand, purchase_price, volume                                                                 |

星型架構設計

                           ┌─────────────────────────────┐
                           │        dim_product          │
                           │ (brand, desc, size, class,  │
                           │          volume)            │
                           └──────────────┬──────────────┘
                                          │
                                          │
        ┌─────────────────┐     ┌──────────────────────┐     ┌─────────────────┐
        │    dim_store    │     │    fact_inventory    │     │    dim_vendor   │
        │ (store_id,      │◄────┤                      ├────►│ (vendor_no,     │
        │  store_name)    │     │ • sales_qty          │     │  vendor_name)   │
        └─────────────────┘     │ • sales_dollars      │     └─────────────────┘
                                │ • purchase_qty       │
        ┌─────────────────┐     │ • beg_inventory      │     ┌─────────────────┐
        │    dim_date     │     │ • end_inventory      │     │  dim_category   │
        │ (date, year,    ├────►│ • excise_tax         │◄────┤ (class_id,      │
        │  month, qtr)    │     │ • purchase_price     │     │  class_name)    │
        └─────────────────┘     └──────────────────────┘     └─────────────────┘

專案目錄結構

```
04_Inventory_Performance_Analysis/
├── data/
│   ├── raw_review/          ✅ 已存在 (100-row previews)
│   └── raw/                 (完整 CSV，不上傳至 Git)
├── sql/
│   ├── 01_ddl/
│   │   ├── 01_create_raw_tables.sql
│   │   ├── 02_create_staging_tables.sql
│   │   └── 03_create_warehouse_tables.sql
│   ├── 02_elt/
│   │   ├── 01_load_raw.sql
│   │   ├── 02_transform_staging.sql
│   │   └── 03_load_warehouse.sql
│   ├── 03_quality/
│   │   └── 01_data_quality_checks.sql
│   └── 04_analysis/
│       ├── 01_inventory_turnover.sql
│       ├── 02_dsi_stockout.sql
│       └── 03_reorder_point.sql
├── powerbi/
│   └── inventory_dashboard.pbix
└── docs/
    └── data_dictionary.md
```


資料清洗校對清單

| 檢查項目            | 邏輯                                | 處理方式                             |
| --------------- | --------------------------------- | -------------------------------- |
| 金額一致性           | sales_dollars ≠ sales_price × qty | dq_flag = 'AMOUNT_MISMATCH' 標記排除 |
| 負數數量            | sales_quantity ≤ 0                | 標記 INVALID_QTY                   |
| inventory_id 格式 | 必須含 3 個 _ 分隔段                     | SPLIT_PART 前先 REGEXP 驗證          |
| 孤兒庫存            | 有 beg/end 但無銷售紀錄                  | LEFT JOIN 保留，標記為潛在呆滯             |
| 重複交易            | 同 inventory_id + sales_date + qty | ROW_NUMBER() 去重                  |
| vendor_name 空白  | TRIM 後為空                          | 用 vendor_no join 補全              |

raw data 載入發現 volume = 'Unknown' 將來無法 ::NUMERIC，必須先決定轉換策略變 NULL